Vamos a extraer los datos de la Espectroscopia de Impedancia Electroquímica de los archivos.

In [22]:
import os
import shutil
import re
import glob
from pathlib import Path
import numpy as np
from bateria import Bateria

In [23]:
import csv
import re
from pathlib import Path

def build_zcurve_matrix(eis_dir: str = "data/raw/eis", output_csv: str = "data/processed/matriz_zcurve.csv"):
    base = Path(eis_dir)
    
    # Buscar todos los archivos .DTA (ignorando mayúsculas/minúsculas en la extensión)
    todos_los_archivos = list(base.glob("*.[dD][tT][aA]"))
    
    if not todos_los_archivos:
        print(f"No se encontraron archivos .DTA en '{base}'. Verifica la ruta.")
        return

    # Extraer el número de batería del nombre (ej. EIS_B1 -> 1) y guardar como tupla (numero, ruta)
    archivos_validos = []
    for archivo in todos_los_archivos:
        # Busca la estructura 'EIS_B' seguida de dígitos
        match = re.search(r'EIS_B(\d+)', archivo.name, re.IGNORECASE)
        if match:
            numero = int(match.group(1)) # El número capturado
            archivos_validos.append((numero, archivo))
        else:
            print(f"Omitiendo '{archivo.name}' — no coincide con el formato 'EIS_B#'.")

    # Ordenar los archivos de menor a mayor según el número de batería
    archivos_validos.sort(key=lambda x: x[0])

    if not archivos_validos:
        print("No hay archivos válidos para procesar.")
        return

    todas_las_filas = []
    encabezados_guardados = False

    # Iteramos sobre la lista ya ordenada de (numero, archivo)
    for numero, archivo_dta in archivos_validos:
        try:
            capturando = False
            filas_bateria = []
            
            with open(archivo_dta, mode="r", encoding="latin-1", errors="replace") as f:
                for linea in f:
                    if linea.startswith("ZCURVE") and "TABLE" in linea:
                        capturando = True
                        continue
                    
                    if capturando:
                        linea_limpia = linea.strip()
                        if not linea_limpia:
                            continue
                        
                        columnas = [col.strip() for col in linea_limpia.split('\t') if col.strip()]
                        if columnas:
                            filas_bateria.append(columnas)
            
            if filas_bateria:
                # 1. Configurar los encabezados globales una sola vez
                if not encabezados_guardados:
                    encabezados_globales = ["Bateria_Num"] + filas_bateria[0]
                    todas_las_filas.append(encabezados_globales)
                    encabezados_guardados = True
                
                # 2. Extraer los datos numéricos
                # Ignoramos index 0 (nombres de columnas) e index 1 (unidades de medida)
                for fila in filas_bateria[2:]:
                    # Cambiamos la coma por punto
                    fila_numerica = [val.replace(',', '.') for val in fila]
                    
                    # Insertamos el número de batería al principio de la fila
                    todas_las_filas.append([numero] + fila_numerica)

        except Exception as e:
            print(f"Error procesando '{archivo_dta.name}': {e} — omitiendo.")

    # --- Grabación de la matriz en CSV ---
    try:
        # Nos aseguramos de que la carpeta de destino exista antes de guardar
        Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_csv, mode="w", newline="", encoding="utf-8") as csv_file:
            writer = csv.writer(csv_file, delimiter=',')
            writer.writerows(todas_las_filas)
            
        print(f"\n¡Matriz ZCURVE guardada exitosamente en: '{output_csv}'!")
    except Exception as e:
        print(f"\nError al guardar el archivo CSV: {e}")

# Ejemplo de uso:
# build_zcurve_matrix()

In [24]:
build_zcurve_matrix()


¡Matriz ZCURVE guardada exitosamente en: 'data/processed/matriz_zcurve.csv'!


In [25]:
import pandas as pd
import numpy as np

def actualizar_impedancia(archivo_zcurve="data/processed/matriz_zcurve.csv", 
                          archivo_baterias="data/processed/matriz_baterias.csv", 
                          output_csv="data/processed/matriz_baterias.csv"):
    
    print("Cargando datos...")
    # 1. Cargar los archivos CSV
    df_zcurve = pd.read_csv(archivo_zcurve)
    df_baterias = pd.read_csv(archivo_baterias)

    # Buscar automáticamente el nombre exacto de las columnas (por si tienen espacios o unidades como 'Zreal (ohm)')
    col_zreal = [col for col in df_zcurve.columns if 'Zreal' in col][0]
    col_zimag = [col for col in df_zcurve.columns if 'Zimag' in col][0]

    # 2. Encontrar el Zreal donde Zimag es más cercano a 0
    # Calculamos el valor absoluto de Zimag para encontrar la distancia al cero
    df_zcurve['Zimag_abs'] = df_zcurve[col_zimag].abs()

    # Buscamos el índice (fila) del valor mínimo de 'Zimag_abs' para CADA batería
    indices_optimos = df_zcurve.groupby('Bateria_Num')['Zimag_abs'].idxmin()

    # Filtramos el dataframe original usando esos índices para quedarnos solo con los cruces por cero
    df_cruces = df_zcurve.loc[indices_optimos, ['Bateria_Num', col_zreal]]

    # 3. Cruzar la información con la matriz de baterías
    # Creamos un "diccionario" virtual que asocia: {Numero_Bateria : Valor_Zreal}
    mapa_impedancias = dict(zip(df_cruces['Bateria_Num'], df_cruces[col_zreal]))

    # Actualizamos la columna "Impedancia" buscando el "Numero" en nuestro mapa
    # Si alguna batería no tiene datos de impedancia, mantenemos el valor original (0) con fillna()
    df_baterias['Impedancia'] = df_baterias['Numero'].map(mapa_impedancias).fillna(df_baterias['Impedancia'])

    # 4. Guardar el nuevo CSV
    df_baterias.to_csv(output_csv, index=False)
    print(f"¡Listo! Se actualizaron las impedancias y el archivo se guardó como: '{output_csv}'")
    
    return df_baterias


In [29]:
actualizar_impedancia()

Cargando datos...
¡Listo! Se actualizaron las impedancias y el archivo se guardó como: 'data/processed/matriz_baterias.csv'


,Numero,SoH(Ah),Impedancia
0,1,0.335735,0.086353
1,2,0.340149,0.170307
2,3,0.361670,0.169233
3,4,0.331654,0.087558
4,5,0.302161,0.171626
5,6,0.345978,0.152892
6,7,0.339422,0.169030
7,8,0.287277,0.210809
8,9,0.342146,0.089985
9,10,0.340374,0.136096
